# Comparing Weight-based and Weightless Neural Networks for Handwritten Digit Recognition

In [1]:
!pip install torch torchvision numpy

In [2]:
# 1. Uninstall any broken pybind11 versions just in case
!pip uninstall -y pybind11 wisardpkg

# 2. Force the installation of an older, compatible pybind11 (v2.x)
!pip install "pybind11<3.0"

# 3. Install wisardpkg without using the cache
!pip install --no-cache-dir wisardpkg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.3/243.3 kB 10.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.5/126.5 kB 42.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for wisardpkg: filename=wisardpkg-1.6.3-cp312-cp312-linux_x86_64.whl size=3331010 sha256=4a897ca85fedc3a0350f618d7fd494ac270e3a84de9fe72eb7c0acfddc840459
  Stored in directory: /tmp/pip-ephem-wheel-cache-8vt1d8t2/wheels/ac/a2/68/6af3d7fbcb1c34042a3a59650c77bec73ab7dcc1048750e464
Successfully built wisardpkg


- In standard model, normalize data to mean and standard deviation
- In WiSARD, binarize it
    - Thresholding: anything above 0.5 = 1, anything below 0.5 = 0

In [3]:
import time
import torch
from torchvision import datasets, transforms
import wisardpkg as wnn
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support

In [4]:
# 1. Load the MNIST dataset w. Torch
transform = transforms.Compose([
    transforms.ToTensor(),
    # WiSARD needs 1D binary vectors, so flatten 28x288 to 784
    transforms.Lambda(lambda x: x.view(-1)),
])

train_set = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_set = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# 2. Binarize data w. threshold 0.5
# WiSARD requires lists of ints or np arrays
# Using 127 because [0, 255] and 127 = 0.5 * 255
X_train = (train_set.data.view(-1, 784).numpy() > 127).astype(int).tolist()
y_train = [str(label) for label in train_set.targets.numpy()]

X_test = (test_set.data.view(-1, 784).numpy() > 127).astype(int).tolist()
y_test = test_set.targets.numpy()

100%|██████████| 9.91M/9.91M [00:00<00:00, 135MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 45.3MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 108MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 10.1MB/s]


In [5]:
# Implement WiSARD

# Define parameters
addressSize = 10 # How many bits / RAM neuron
bleaching = True # Handles ties bw classes

# Initialize model
wnn_wisard = wnn.Wisard(addressSize, bleachingActivated=bleaching)

# Training - one pass, no backprop
print("Training WiSARD...")
start_time = time.time()
wnn_wisard.train(X_train, y_train)
training_time = time.time() - start_time

# Classification / testing
print("Testing WiSARD...")
precictions = wnn_wisard.classify(X_test)

Training WiSARD...
Testing WiSARD...


In [6]:
# wisardpkg returns string labels, so convert to int
y_pred = np.array([int(p) for p in precictions])

# Accuracy
accuracy = np.mean(y_pred == y_test)
print(f"WiSARD Accuracy: {accuracy * 100:.2f}%")

WiSARD Accuracy: 79.02%


In [7]:
test_acc = accuracy_score(y_test, y_pred)
macro_prec, macro_rec, macro_f1, _ = precision_recall_fscore_support(y_test, y_pred, average='macro')

class_report = classification_report(y_test, y_pred, digits=4)
conf_matrix = confusion_matrix(y_test, y_pred)

In [8]:
print(f"Best validation accuracy: N/A (WiSARD trains in a single pass / one-shot learning, so no validation set.)")
print(f"Training time: {training_time:.2f} seconds\n")

print("Test Results")
print(f"Test Loss: N/A (WiSARD is RAM-based)")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Macro Precision: {macro_prec:.4f}")
print(f"Macro Recall: {macro_rec:.4f}")
print(f"Macro F1-score: {macro_f1:.4f}\n")

print("Classification Report:")
print(class_report)

print("Confusion Matrix:")
print(conf_matrix)

Best validation accuracy: N/A (WiSARD trains in a single pass / one-shot learning, so no validation set.)
Training time: 1.52 seconds

Test Results
Test Loss: N/A (WiSARD is RAM-based)
Test Accuracy: 0.7902
Macro Precision: 0.8118
Macro Recall: 0.7906
Macro F1-score: 0.7940

Classification Report:
              precision    recall  f1-score   support

           0     0.9361    0.9112    0.9235       980
           1     1.0000    0.7489    0.8564      1135
           2     0.7203    0.8886    0.7957      1032
           3     0.7538    0.6911    0.7211      1010
           4     0.8714    0.7179    0.7873       982
           5     0.6611    0.7500    0.7027       892
           6     0.9423    0.8518    0.8947       958
           7     0.9118    0.7646    0.8317      1028
           8     0.5462    0.8070    0.6515       974
           9     0.7750    0.7750    0.7750      1009

    accuracy                         0.7902     10000
   macro avg     0.8118    0.7906    0.7940     100